# Reranking via OpenRouter /api/v1/rerank Endpoint

Using OpenRouter's dedicated cross-encoder reranking API for efficient document reranking.

This notebook tests reranking strategies using OpenRouter's dedicated endpoint:
- Cohere Rerank v3.5 (recommended, balanced)
- Cohere Rerank 4-Pro (latest, best quality)
- NVIDIA Llama Nemotron Rerank (free, efficient)

## Optimal Baseline (Frozen):
- **Chunking:** Semantic-Level, 192 tokens
- **Retrieval:** Dense MMR, k=8
- **Current Performance:** CR=0.0616, Adh=0.8

## Reranking Models:
1. **No Reranking** - Baseline (control)
2. **Cohere Rerank v3.5** - Balanced, recommended by Cohere
3. **Cohere Rerank 4-Pro** - Latest, best quality
4. **NVIDIA Llama Nemotron** - Free, open source

Goal: Find optimal reranking model via OpenRouter's dedicated endpoint

## Step 1: Install Dependencies

In [10]:
!pip install -q python-dotenv datasets tiktoken langchain-core langchain-text-splitters langchain-huggingface langchain-chroma langchain-openai nltk requests


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Step 2: Import Libraries

In [11]:
import os
import json
import re
import numpy as np
import pandas as pd
import tiktoken
import tempfile
import time
import requests
from dotenv import load_dotenv
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
import nltk
from typing import List, Dict, Tuple

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

nltk.download('punkt_tab', quiet=True)

load_dotenv()
openrouter_token = os.environ.get('OPENROUTER_TOKEN')

print("✓ All imports successful")

✓ All imports successful


## Step 3: Load Dataset & Setup

In [ ]:
import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.data_loading import load_rag_bench_data
from ragbench_lib.models import get_embedding_model, get_generation_llm, get_judge_llm
from ragbench_lib.generation_prompt import RAG_GENERATION_PROMPT

print("Loading dataset...")
DATASET_NAME = "delucionqa"  # feeds both load_rag_bench_data() and vector store naming below
docs_df = load_rag_bench_data(DATASET_NAME, num_samples=50)
print(f"✓ Loaded {len(docs_df)} documents from {docs_df['row_id'].nunique()} unique questions")

# Setup models
embedding_model = get_embedding_model(openrouter_token)

llm_base = get_generation_llm(openrouter_token)

llama_judge = get_judge_llm(openrouter_token)

prompt = RAG_GENERATION_PROMPT

print("✓ Models configured")


## Step 4: Semantic-Level Chunking (Frozen)

In [ ]:
from ragbench_lib.chunking import count_tokens, get_sentences, create_semantic_chunks

print("Preparing semantic chunks (192t)...")
documents = create_semantic_chunks(docs_df, target_tokens=192)
print(f"✓ Created {len(documents)} semantic chunks")


## Step 5: Dense MMR Retriever (Frozen)

In [ ]:
from ragbench_lib.retrievers import DenseMMRRetriever

print("Creating Dense MMR retriever...")
retriever = DenseMMRRetriever(documents, embedding_model, k=8, dataset_name=DATASET_NAME)
print("✓ Dense MMR retriever ready")


## Step 6: OpenRouter Reranking API Client

In [15]:
class BaseReranker:
    """Base class for rerankers"""
    def rerank(self, query: str, documents: List[Document], top_k: int = 8) -> List[Document]:
        raise NotImplementedError


class NoReranker(BaseReranker):
    """No reranking - baseline control"""
    def rerank(self, query: str, documents: List[Document], top_k: int = 8) -> List[Document]:
        return documents[:top_k]


class OpenRouterRerankerAPI(BaseReranker):
    """Reranker using OpenRouter's dedicated /api/v1/rerank endpoint"""
    def __init__(self, model_name: str, api_key: str):
        self.model_name = model_name
        self.api_key = api_key
        self.endpoint = "https://openrouter.ai/api/v1/rerank"
    
    def rerank(self, query: str, documents: List[Document], top_k: int = 8) -> List[Document]:
        """Rerank documents using OpenRouter's rerank endpoint"""
        
        # Extract document texts
        doc_texts = [doc.page_content for doc in documents]
        
        # Prepare request
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": self.model_name,
            "query": query,
            "documents": doc_texts,
            "top_n": top_k
        }
        
        try:
            response = requests.post(self.endpoint, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            
            result = response.json()
            
            # Extract results and sort by relevance score
            if 'results' in result:
                results = result['results']
                # Sort by relevance_score (descending)
                results = sorted(results, key=lambda x: x.get('relevance_score', 0), reverse=True)
                
                # Get top_k indices
                ranked_indices = [r['index'] for r in results[:top_k]]
                return [documents[i] for i in ranked_indices]
            else:
                # Fallback if response format unexpected
                return documents[:top_k]
        except Exception as e:
            print(f"    Reranking error: {str(e)}")
            # Fallback to original order
            return documents[:top_k]


print("✓ Reranker classes defined")

✓ Reranker classes defined


## Step 7: TRACe Evaluation Metrics

In [ ]:
from ragbench_lib.chunking import get_sentences
from ragbench_lib.trace_eval import (
    format_documents_with_keys,
    annotate_response_for_metrics as _annotate_response_for_metrics,
    compute_context_relevance,
    compute_utilization,
    compute_completeness,
    compute_adherence,
)


def annotate_response_for_metrics(documents, question, response):
    """Annotate a response using this notebook's judge LLM (llama_judge)."""
    return _annotate_response_for_metrics(llama_judge, documents, question, response)


print("✓ TRACe metric functions ready (ragbench_lib.trace_eval)")

## Step 8: Experiment Runner

In [17]:
def run_reranking_experiment(reranker_name: str, reranker, docs_df: pd.DataFrame, retriever, llm, prompt, num_samples=5):
    """Run experiment for a specific reranking strategy"""
    print(f"  {reranker_name:50s}...", end=" ", flush=True)
    
    try:
        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)
        
        rag_chain = ({"context": lambda x: format_docs(reranker.rerank(x, retriever.retrieve(x))), "question": lambda x: x} 
                     | prompt | llm | StrOutputParser())
        
        unique_samples = docs_df.drop_duplicates(subset=['row_id']).head(num_samples).reset_index(drop=True)
        results = []
        
        for i, row in unique_samples.iterrows():
            try:
                question = row['question']
                my_response = rag_chain.invoke(question)
                retrieved_docs = retriever.retrieve(question)
                reranked_docs = reranker.rerank(question, retrieved_docs, top_k=8)
                retrieved_texts = [doc.page_content for doc in reranked_docs]
                
                annotation = annotate_response_for_metrics(retrieved_texts, question, my_response)
                
                if annotation['success']:
                    results.append({
                        'context_relevance': compute_context_relevance(retrieved_texts, annotation),
                        'utilization': compute_utilization(retrieved_texts, annotation),
                        'completeness': compute_completeness(annotation),
                        'adherence': compute_adherence(annotation),
                    })
            except Exception as e:
                pass
        
        if results:
            avg_metrics = {
                'strategy': reranker_name,
                'context_relevance': np.mean([r['context_relevance'] for r in results]),
                'utilization': np.mean([r['utilization'] for r in results]),
                'completeness': np.mean([r['completeness'] for r in results]),
                'adherence': np.mean([r['adherence'] for r in results]),
            }
            print(f"✓ (CR: {avg_metrics['context_relevance']:.4f})")
            return avg_metrics
        else:
            print("✗ No results")
            return None
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
        return None

print("✓ Experiment runner ready")

✓ Experiment runner ready


## Step 9: Run OpenRouter Reranking Experiments

In [18]:
print("\n" + "="*120)
print("OPENROUTER CROSS-ENCODER RERANKING COMPARISON")
print("Using dedicated /api/v1/rerank endpoint")
print("="*120 + "\n")

reranking_results = []
baseline_cr = 0.0616

# R0: No reranking (baseline control)
print("Testing baseline...")
reranker = NoReranker()
result = run_reranking_experiment("R0: No Reranking (Baseline - Dense MMR)", reranker, docs_df, retriever, llm_base, prompt, num_samples=5)
if result:
    reranking_results.append(result)
    baseline_cr = result['context_relevance']
    print(f"  (Baseline CR established: {baseline_cr:.4f})")

print("\nTesting OpenRouter cross-encoder rerankers...\n")

# R1: Cohere Rerank v3.5 (recommended, balanced)
try:
    reranker = OpenRouterRerankerAPI("cohere/rerank-v3.5", openrouter_token)
    result = run_reranking_experiment("R1: Cohere Rerank v3.5 (Recommended)", reranker, docs_df, retriever, llm_base, prompt, num_samples=5)
    if result:
        reranking_results.append(result)
except Exception as e:
    print(f"  R1: Cohere Rerank v3.5        ... ✗ Error: {str(e)[:50]}")

# R2: Cohere Rerank 4-Pro (latest, best quality)
try:
    reranker = OpenRouterRerankerAPI("cohere/rerank-4-pro", openrouter_token)
    result = run_reranking_experiment("R2: Cohere Rerank 4-Pro (Best Quality)", reranker, docs_df, retriever, llm_base, prompt, num_samples=5)
    if result:
        reranking_results.append(result)
except Exception as e:
    print(f"  R2: Cohere Rerank 4-Pro       ... ✗ Error: {str(e)[:50]}")

# R3: NVIDIA Llama Nemotron (free, efficient)
try:
    reranker = OpenRouterRerankerAPI("nvidia/llama-nemotron-rerank-vl-1b-v2:free", openrouter_token)
    result = run_reranking_experiment("R3: NVIDIA Nemotron (Free, Efficient)", reranker, docs_df, retriever, llm_base, prompt, num_samples=5)
    if result:
        reranking_results.append(result)
except Exception as e:
    print(f"  R3: NVIDIA Nemotron (Free)    ... ✗ Error: {str(e)[:50]}")

print("\n" + "="*120)
print("RERANKING RESULTS")
print("="*120)

if reranking_results:
    df_results = pd.DataFrame(reranking_results)
    display(df_results)
    
    best_idx = df_results['context_relevance'].idxmax()
    best_result = df_results.iloc[best_idx]
    
    print("\n" + "-"*120)
    print(f"🏆 BEST RERANKER: {best_result['strategy']}")
    print("-"*120)
    print(f"  Context Relevance:  {best_result['context_relevance']:.4f}")
    print(f"  Utilization:        {best_result['utilization']:.4f}")
    print(f"  Completeness:       {best_result['completeness']:.4f}")
    print(f"  Adherence:          {best_result['adherence']:.4f}")
    print("-"*120)
    
    print("\nFULL RANKING:")
    df_ranked = df_results.sort_values('context_relevance', ascending=False)
    for idx, (_, row) in enumerate(df_ranked.iterrows(), 1):
        medal = "🥇" if idx == 1 else "🥈" if idx == 2 else "🥉" if idx == 3 else "  "
        improvement = ((row['context_relevance'] - baseline_cr) / baseline_cr * 100) if baseline_cr > 0 else 0
        print(f"{medal} {idx}. {row['strategy']:55s} | CR: {row['context_relevance']:.4f} ({improvement:+.1f}%) | Util: {row['utilization']:.4f} | Compl: {row['completeness']:.4f} | Adh: {row['adherence']:.4f}")
else:
    print("No results collected")


OPENROUTER CROSS-ENCODER RERANKING COMPARISON
Using dedicated /api/v1/rerank endpoint

Testing baseline...
  R0: No Reranking (Baseline - Dense MMR)           ... ✓ (CR: 0.1700)
  (Baseline CR established: 0.1700)

Testing OpenRouter cross-encoder rerankers...

  R1: Cohere Rerank v3.5 (Recommended)              ... ✓ (CR: 0.0798)
  R2: Cohere Rerank 4-Pro (Best Quality)            ... ✓ (CR: 0.1378)
  R3: NVIDIA Nemotron (Free, Efficient)             ... ✓ (CR: 0.0780)

RERANKING RESULTS


,strategy,context_relevance,utilization,completeness,adherence
0,R0: No Reranking (Baseline - Dense MMR),0.16996,0.07124,0.58572,0.6
1,R1: Cohere Rerank v3.5 (Recommended),0.07980,0.04890,0.69524,1.0
2,R2: Cohere Rerank 4-Pro (Best Quality),0.13780,0.05992,0.75142,1.0
3,"R3: NVIDIA Nemotron (Free, Efficient)",0.07796,0.04624,0.61846,1.0



------------------------------------------------------------------------------------------------------------------------
🏆 BEST RERANKER: R0: No Reranking (Baseline - Dense MMR)
------------------------------------------------------------------------------------------------------------------------
  Context Relevance:  0.1700
  Utilization:        0.0712
  Completeness:       0.5857
  Adherence:          0.6000
------------------------------------------------------------------------------------------------------------------------

FULL RANKING:
🥇 1. R0: No Reranking (Baseline - Dense MMR)                 | CR: 0.1700 (+0.0%) | Util: 0.0712 | Compl: 0.5857 | Adh: 0.6000
🥈 2. R2: Cohere Rerank 4-Pro (Best Quality)                  | CR: 0.1378 (-18.9%) | Util: 0.0599 | Compl: 0.7514 | Adh: 1.0000
🥉 3. R1: Cohere Rerank v3.5 (Recommended)                    | CR: 0.0798 (-53.0%) | Util: 0.0489 | Compl: 0.6952 | Adh: 1.0000
   4. R3: NVIDIA Nemotron (Free, Efficient)                   | C

## Step 10: Recommendations

In [19]:
if reranking_results:
    df_results = pd.DataFrame(reranking_results).sort_values('context_relevance', ascending=False)
    baseline_row = df_results[df_results['strategy'].str.contains('No Reranking')]
    
    if len(baseline_row) > 0:
        baseline_cr = baseline_row.iloc[0]['context_relevance']
    else:
        baseline_cr = 0.0616
    
    print("\n" + "="*120)
    print("OPENROUTER CROSS-ENCODER RERANKING RECOMMENDATIONS")
    print("="*120)
    
    best = df_results.iloc[0]
    if 'No Reranking' not in best['strategy']:
        improvement = ((best['context_relevance'] - baseline_cr) / baseline_cr * 100) if baseline_cr > 0 else 0
    else:
        improvement = 0
    
    print(f"\n📊 Results Summary:")
    print(f"  Best Strategy: {best['strategy']}")
    print(f"  Context Relevance: {best['context_relevance']:.4f}")
    print(f"  Improvement vs Baseline: {improvement:+.1f}%")
    print(f"  Adherence: {best['adherence']:.4f}")
    print(f"  Completeness: {best['completeness']:.4f}")
    print(f"  Utilization: {best['utilization']:.4f}")
    
    print(f"\n💡 Key Insights:")
    print(f"  • OpenRouter's dedicated /api/v1/rerank endpoint: efficient cross-encoder reranking")
    print(f"  • Cohere Rerank: Industry-standard, production-ready")
    print(f"  • NVIDIA Nemotron: Free, open-source alternative")
    print(f"  • Trade-off: Improved quality vs. latency (per-document ranking)")
    
    print(f"\n🎯 Deployment Decision:")
    if improvement > 5:
        print(f"  ✅ RECOMMENDED: Deploy {best['strategy']}")
        print(f"     • Significant improvement (+{improvement:.1f}%) justifies latency cost")
        print(f"     • Configuration: Dense MMR (k=8) → {best['strategy'].split(':')[1].strip()} → Generator")
    elif improvement > -5:
        print(f"  ⚠️  OPTIONAL: {best['strategy']}")
        print(f"     • Small improvement ({improvement:+.1f}%)")
        print(f"     • Consider cost/latency trade-off")
        print(f"     • Recommend: Skip reranking, focus on generator fine-tuning")
    else:
        print(f"  ❌ NOT RECOMMENDED")
        print(f"     • All rerankers decreased performance")
        print(f"     • Dense MMR baseline is optimal")
    
    print(f"\n📋 Next Steps:")
    print(f"  1. Implement generator fine-tuning with Dr strategy (random documents)")
    print(f"  2. Expected generator FT improvement: +78% context relevance")
    print(f"  3. Create end-to-end pipeline:")
    print(f"     Semantic 192t Chunks → Dense MMR", end="")
    if improvement > 0:
        print(f" → Reranking", end="")
    print(f" → Fine-tuned Generator")
    print(f"  4. Benchmark full RAG pipeline")
    
    print("\n" + "="*120)


OPENROUTER CROSS-ENCODER RERANKING RECOMMENDATIONS

📊 Results Summary:
  Best Strategy: R0: No Reranking (Baseline - Dense MMR)
  Context Relevance: 0.1700
  Improvement vs Baseline: +0.0%
  Adherence: 0.6000
  Completeness: 0.5857
  Utilization: 0.0712

💡 Key Insights:
  • OpenRouter's dedicated /api/v1/rerank endpoint: efficient cross-encoder reranking
  • Cohere Rerank: Industry-standard, production-ready
  • NVIDIA Nemotron: Free, open-source alternative
  • Trade-off: Improved quality vs. latency (per-document ranking)

🎯 Deployment Decision:
  ⚠️  OPTIONAL: R0: No Reranking (Baseline - Dense MMR)
     • Small improvement (+0.0%)
     • Consider cost/latency trade-off
     • Recommend: Skip reranking, focus on generator fine-tuning

📋 Next Steps:
  1. Implement generator fine-tuning with Dr strategy (random documents)
  2. Expected generator FT improvement: +78% context relevance
  3. Create end-to-end pipeline:
     Semantic 192t Chunks → Dense MMR → Fine-tuned Generator
  4. Be